# 神经网络结构与训练框架


### 🧩 第1节：nn.Module 与多层感知机（MLP）结构
🎯 学习目标

通过这一节，你将掌握：

1. torch.nn.Module 的基本用法

2. 如何定义神经网络层（线性层 + 激活函数）

3. 前向传播 forward() 的机制

4. 模型对象的使用与参数访问

#### 最小可运行的 MLP 模型

In [3]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MLP()
print(model)

x = torch.tensor([[1.0, 2.0]])
y = model(x)
print(y)
        

MLP(
  (fc1): Linear(in_features=2, out_features=4, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=4, out_features=1, bias=True)
)
tensor([[0.3239]], grad_fn=<AddmmBackward0>)


非常好 👍！
现在你已经能成功定义并运行一个最基础的多层感知机（MLP）模型了。
接下来我们进入 下一步学习阶段 —— 在这个阶段，我们将让模型学会拟合数据，也就是让它通过训练自动调整参数。

#### 训练一个简单的 MLP
目标：让模型学会拟合一个二维输入到标量输出的函数

我们手动生成一些训练数据，例如：

In [ ]:
# 1. 准备数据

import torch

x = torch.rand(100, 2)                  # (100, 2)
y = 3 * x[:, 0] + 2 * x[:, 1] + 1       # (100, )
y = y.unsqueeze(1)                      # (100, 1)


# 2. 定义模型

import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MLP()


# 3. 定义损失函数和优化器

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


# 4. 训练模型

for epoch in range(200):
    # 前向传播
    y_pred = model(x)
    loss = criterion(y_pred, y)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.6f}")
        

# 5. 查看效果

print("训练后的预测")
test = torch.tensor([[0.5, 0.5]])
print("模型预测结果：", model(test).item())
print("真实结果", 3 * 0.5 + 2 * 0.5 + 1)

Epoch   0 | Loss: 14.097671
Epoch  20 | Loss: 0.168003
Epoch  40 | Loss: 0.017898
Epoch  60 | Loss: 0.001132
Epoch  80 | Loss: 0.000076
Epoch 100 | Loss: 0.000021
Epoch 120 | Loss: 0.000018
Epoch 140 | Loss: 0.000018
Epoch 160 | Loss: 0.000018
Epoch 180 | Loss: 0.000017
训练后的预测
模型预测结果： 3.5011627674102783
真实结果 3.5


### 🧩 第2节：在 MNIST 数据集上实践

#### LeNet

In [1]:
# 一、导入依赖和数据集

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备：", device)

# 1. 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307, ), (0.3081, ))
])

# 2. 下载训练集和测试集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

# 3. 使用dataloader按批加载
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


# 二、定义LeNet网络结构

class LeNet(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = LeNet().to(device)
print(model)


# 三、定义损失函数和优化器

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)


# 四、训练

for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")
            
    print(f"Epoch {epoch} 平均Loss: {running_loss / len(train_loader):.4f}")
    
    
# 五、在测试集上评估

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"在测试集上准确率：{100 * correct / total:.2f}%")

使用设备： cuda
LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)
Epoch [1], Step [0/938], Loss: 2.2937
Epoch [1], Step [100/938], Loss: 0.1862
Epoch [1], Step [200/938], Loss: 0.1807
Epoch [1], Step [300/938], Loss: 0.3618
Epoch [1], Step [400/938], Loss: 0.1056
Epoch [1], Step [500/938], Loss: 0.0414
Epoch [1], Step [600/938], Loss: 0.2296
Epoch [1], Step [700/938], Loss: 0.1283
Epoch [1], Step [800/938], Loss: 0.1252
Epoch [1], Step [900/938], Loss: 0.1786
Epoch 1 平均Loss: 0.2730
Epoch [2], Step [0/938], Loss: 0.1015
Epoch [2], Step [100/938], Loss: 0.0042
Epoch [2], Step [200/938], Loss: 0.0222
Epoch [2], Step [300/938], Loss: 0.1583
Epoch [2], Step [400/938], Loss: 0.0295
Epo

改进LeNet: 加Batch Normalization 和 Dropout

Batch Normalization: 在每个通道上做标准化，使分布为正态分布

Dropout: 在训练时使一些神经元输出置0，其他神经元输出相应增大，以实现避免对单一神经元输出的依赖。测试时就不会置零

In [1]:
# 一、导入依赖和数据集

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备：", device)

# 1. 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307, ), (0.3081, ))
])

# 2. 下载训练集和测试集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

# 3. 使用dataloader按批加载
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


# 二、定义LeNet网络结构

class LeNet(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.bn1 = nn.BatchNorm2d(6)
        self.pool = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.bn2 = nn.BatchNorm2d(16)
        
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(120, 84)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

model = LeNet().to(device)
print(model)


# 三、定义损失函数和优化器

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)


# 四、训练

for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")
            
    print(f"Epoch {epoch} 平均Loss: {running_loss / len(train_loader):.4f}")
    
    
# 五、在测试集上评估

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"在测试集上准确率：{100 * correct / total:.2f}%")

使用设备： cuda
LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (bn1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (dropout1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)
Epoch [1], Step [0/938], Loss: 2.3255
Epoch [1], Step [100/938], Loss: 0.6623
Epoch [1], Step [200/938], Loss: 0.5047
Epoch [1], Step [300/938], Loss: 0.4174
Epoch [1], Step [400/938], Loss: 0.2736
Epoch [1], Step [500/938], Loss: 0.1729
Epoch [1], Step [600/938], Loss: 0.1349
Epoch [1], Step [700/938], Loss: 0.3492
Epoch [1], Step [800/938], Loss: 0.2762

#### AlexNet

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
model = AlexNet(num_classes=1000)
x = torch.randn(1, 3, 224, 224)
y = model(x)
print("输出形状：", y.shape)

输出形状： torch.Size([1, 1000])


非常棒 👍，你整体的理解已经非常接近标准答案了！我先逐条点评你的回答，然后给出**标准答案**，最后我们马上进入你说的下一步——写 AlexNet 预训练权重推理代码。

---

### ✅ 你的回答点评 + 标准答案

---

#### **1️⃣ 为什么第一层卷积使用 stride=4？**

> 🟢 你的回答：「因为第一层图像还很大，需要迅速降低图像大小，提取特征。」

✅ **点评：**
完全正确！这句话非常准确地反映了设计初衷。
再补充一点细节：AlexNet 的输入是 **224×224×3**，第一层使用 **11×11 卷积核 + stride=4**，能让特征图在第一步就缩小为 **(224−11)/4+1≈55**，显著减小计算量。
此外，这样还能让感受野迅速覆盖更大区域，提取粗粒度特征（比如边缘、色块）。

📘 **标准回答：**

> 第一层使用 stride=4 是为了快速降低输入特征图的空间尺寸，减少计算量，同时扩大感受野，从整幅图像中提取低级但全局的特征（如边缘和颜色块）。

---

#### **2️⃣ 为什么卷积核越往后越小？**

> 🟢 你的回答：「因为特征图像也越来越小，小的卷积核能避免忽略关键信息。」

✅ **点评：**
非常好，逻辑正确！可以补充一点更深层的原因：
早期大卷积核（如 11×11、5×5）用于捕获大范围特征，而后期小卷积核（3×3）让网络能堆叠更多层、提取复杂组合特征。

📘 **标准回答：**

> 卷积层越往后，特征图越小，使用小卷积核能更精细地捕捉局部特征，并减少参数量。同时堆叠多个小卷积层的效果相当于大卷积核，但更高效、更具非线性表达能力。

---

#### **3️⃣ Dropout 只在全连接层使用的原因？**

> 🟢 你的回答：「保证特征提取阶段正常，而在根据特征分类的阶段可以训练模型更关注关键特征来进行分类。」

✅ **点评：**
完美！非常接近标准答案。
可以再强调一点：卷积层参数共享，本身就有一定正则化效果，不容易过拟合；而全连接层参数量大、最容易过拟合，所以才单独使用 Dropout。

📘 **标准回答：**

> Dropout 主要用于全连接层，因为卷积层参数共享，特征具有空间相关性，不易过拟合；而全连接层参数多，易记忆训练集，Dropout 能有效提升泛化性能。

---

#### **4️⃣ 为什么选择分层卷积，而不是一个大卷积核？**

> 🟢 你的回答：「分层卷积通过加深深度，能用更小的参数量实现更好的效果。」

✅ **点评：**
非常优秀，这就是核心思想。
补充：多层小卷积相当于一个大卷积核的感受野，但插入非线性激活（ReLU）后能显著提升表达能力。

📘 **标准回答：**

> 多层小卷积堆叠可以扩大感受野、减少参数量、提升非线性表达能力；相比一个大卷积核，它既高效又能更好地分层抽象特征。

---

#### **5️⃣ LocalResponseNorm（LRN）在现代网络中为何被弃用？**

> 🔴 你的回答：「我还不知道这个层是用来干什么的。」

✅ **点评：**
没关系，这个很多人第一次都不了解。我们来讲清楚👇

📘 **标准回答：**

> LRN（局部响应归一化）在 AlexNet 中用于模拟生物神经抑制机制，使响应较大的神经元抑制周围神经元，提高特征对比度。
> 但它带来额外计算开销，而 **Batch Normalization**（BN）出现后能更稳定地加速收敛、提升准确率，因此现代网络几乎不再使用 LRN。

---

### 💡总结一句话

你已经掌握了 AlexNet 的核心思想：

> **通过分层小卷积和非线性激活，快速下采样、减少参数量，同时引入 Dropout 正则化和 ReLU 激活来增强表达能力。**

---

### ✅ 下一步：AlexNet 预训练模型推理（ImageNet）

你说得对，下一步就该实战。
我们将使用 `torchvision.models.alexnet(pretrained=True)` 来做一次 **图片推理实验**。

是否希望我带你写以下完整流程：

1. 加载 AlexNet（预训练权重）
2. 载入并预处理一张图片（用 PIL + transforms）
3. 推理得到 top-5 预测结果（打印类别名）

要不要我直接帮你写出这份可运行代码？


In [8]:
import torch
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO

model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
model.eval()


preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


url = "https://pytorch.org/assets/images/dog.jpg"  # 示例狗狗图
response = requests.get(url)
img = Image.open(BytesIO(response.content))

img_t = preprocess(img)
batch_t = img_t.unsqueeze(0)  # 增加 batch 维度



with torch.no_grad():
    output = model(batch_t)

# 获取预测分数最高的前5个类别
_, indices = torch.sort(output, descending=True)
top5 = indices[0][:5].tolist()

# 获取标签名称
categories = models.AlexNet_Weights.IMAGENET1K_V1.meta["categories"]
for idx in top5:
    print(f"{categories[idx]}: {output[0, idx].item():.4f}")

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /home/boot/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100.0%


Samoyed: 16.8252
wallaby: 15.1769
Pomeranian: 14.3130
Angora: 13.3678
Arctic fox: 12.7614


#### ResNet

In [ ]:
import torch
import torch.nn as nn

# ======= 残差块定义 =======
class BasicBlock(nn.Module):
    expansion = 1  # 输出通道倍数（Bottleneck会是4）

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = downsample  # 当输入输出维度不同时，用这个层调整维度

    def forward(self, x):
        identity = x  # 保存原始输入

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 如果通道或尺寸不同，用downsample调整
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity  # 残差连接
        out = self.relu(out)

        return out


# ======= ResNet主体 =======
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super().__init__()
        self.in_channels = 64

        # 第一层
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # 残差层结构
        self.layer1 = self._make_layer(block, 64,  layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # 全连接层
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        # 如果输入输出不匹配（通道或步幅不同），建立下采样模块
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )

        layers = []
        # 第一层可能需要downsample
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion

        # 余下的层不需要下采样
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


# ======= 构造ResNet18 =======
def resnet18(num_classes=1000):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)


ResNet并没有那么好理解

模型结构

| 阶段           | 输入尺寸      | 卷积结构                           | 输出通道 | 残差块数 | 步长 |
| ------------ | --------- | ------------------------------ | ---- | ---- | -- |
| conv1        | 224×224×3 | 7×7 Conv + BN + ReLU + MaxPool | 64   | —    | 2  |
| layer1       | 56×56     | BasicBlock                     | 64   | 2    | 1  |
| layer2       | 28×28     | BasicBlock                     | 128  | 2    | 2  |
| layer3       | 14×14     | BasicBlock                     | 256  | 2    | 2  |
| layer4       | 7×7       | BasicBlock                     | 512  | 2    | 2  |
| avgpool + fc | —         | 自适应平均池化 + 全连接                  | —    | —    | —  |


layerx调用_make_layer创建

_make_layer 可以根据输入要求创建指定数量和大小的核心模块

核心模块就是所理解的，普通卷积+恒等连接，但是为了这个加法能实现，要把大小调整成一样，二维图像尺寸大小不同或者通道数不同就使用downsample，卷积+batchnorm解决
